In [1]:
# Keep imports and data/output paths relative to the repository root.
import os
import sys
from pathlib import Path
from datetime import datetime
from uuid import uuid4

for repository_dir in (Path.cwd(), *Path.cwd().parents):
    if (repository_dir / ".geofm-root").is_file():
        os.chdir(repository_dir)
        break
else:
    raise FileNotFoundError("Open this notebook from inside the repository.")
if "src" not in sys.path:
    sys.path.insert(0, "src")

EXPERIMENT_NAME = "sen1floods11/terramind/fcn_s1"
NOTEBOOK_PATH = "experiments/sen1floods11/terramind/fcn_s1/01_terramind_on_sen1floods11_training.ipynb"


In [ ]:
%pip install terratorch==1.2.5 torchgeo==0.8.1

#### **Imports**

In [2]:
import torch
import torch.nn as nn
import torchvision.transforms.functional as FT
import torch.nn.functional as F
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import csv
import random
import json

from PIL import Image
from pathlib import Path
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.models.segmentation.fcn import FCNHead
from IPython.display import display


In [3]:
# One-time setup in this notebook kernel: %pip install terratorch==1.2.5 torchgeo==0.8.1
# TerraMind input reference: https://github.com/IBM/terramind/blob/main/configs/terramind_v1_base_sen1floods11.yaml
import hashlib
from importlib.metadata import version
from segmentation_metrics import binary_metrics, metric_table, save_json, save_per_chip_metrics

CROP_SIZE = 120
IGNORE_INDEX = 255
SEED = 42
SCHEMA = "terramind_large_s1_fcn_v1"
PREPROCESSING = {
    "name": "terramind_v1_s1grd", "bands": ["VV", "VH"],
    "mean": [-12.599, -20.293], "std": [5.195, 5.890],
    "nonfinite_raw_replacement": 0.0, "clip": None,
    "patch_size": 16, "padding": "right_bottom_replicate_to_multiple_of_16",
    "crop_size": CROP_SIZE, "ignore_index": IGNORE_INDEX,
}
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
crop_generator = torch.Generator().manual_seed(SEED)
loader_generator = torch.Generator().manual_seed(SEED)


def sha256_file(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def load_split_paths(csv_file, x_dir, y_dir):
    with open(csv_file, newline="") as handle:
        pairs = [(Path(x_dir) / x.strip(), Path(y_dir) / y.strip()) for x, y in csv.reader(handle)]
    missing = [str(p) for pair in pairs for p in pair if not p.is_file()]
    if missing:
        raise FileNotFoundError(f"Missing split inputs or labels: {missing[:5]}")
    return pairs


def audit_splits(index_dir, x_dir, y_dir):
    result, seen = {}, set()
    for split, filename, count in [
        ("train", "flood_train_data.csv", 252), ("valid", "flood_valid_data.csv", 89),
        ("non_bol", "flood_test_data.csv", 90), ("bol", "flood_bolivia_data.csv", 15),
    ]:
        path = Path(index_dir) / filename
        pairs = load_split_paths(path, x_dir, y_dir)
        ids = [y.name.removesuffix("_LabelHand.tif") for x, y in pairs]
        if len(ids) != count or len(set(ids)) != count or seen.intersection(ids):
            raise ValueError(f"Unexpected count, duplicate or split overlap: {split}")
        seen.update(ids)
        result[split] = {"count": count, "csv_sha256": sha256_file(path), "chip_ids": ids}
    return result


def load_image_pair(pair):
    with rasterio.open(pair[0]) as image_src, rasterio.open(pair[1]) as label_src:
        if (image_src.shape != label_src.shape or image_src.crs != label_src.crs
                or not image_src.transform.almost_equals(label_src.transform)):
            raise ValueError(f"Image/label grid mismatch: {pair}")
        image = image_src.read().astype(np.float32)
        mask = label_src.read(1).astype(np.int64)
    # Raw Sen1Floods11 S1 is already VV/VH in dB; no CROMA clipping or scaling.
    image = np.nan_to_num(image, nan=0.0, posinf=0.0, neginf=0.0)
    return image, mask


def prepare_sample(data):
    image, mask = data
    image = torch.as_tensor(image, dtype=torch.float32)
    mask = torch.as_tensor(mask, dtype=torch.long).clone()
    mask[mask == -1] = IGNORE_INDEX
    if image.ndim != 3 or image.shape[0] != 2 or mask.shape != image.shape[-2:]:
        raise ValueError("Expected aligned VV/VH image and two-dimensional mask")
    if not set(torch.unique(mask).tolist()).issubset({0, 1, IGNORE_INDEX}):
        raise ValueError("Unexpected target values")
    mean = image.new_tensor(PREPROCESSING["mean"])[:, None, None]
    std = image.new_tensor(PREPROCESSING["std"])[:, None, None]
    image = (image - mean) / std
    if not torch.isfinite(image).all():
        raise ValueError("Nonfinite normalized input")
    return image, mask


class Sen1Floods11Dataset(Dataset):
    def __init__(self, data_list, preprocess_func):
        self.data_list = data_list
        self.preprocess_func = preprocess_func

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, index):
        return self.preprocess_func(load_image_pair(self.data_list[index]))


def load_terramind(pretrained_path):
    # Check compatibility before TerraTorch imports its optional model factories.
    from importlib.metadata import PackageNotFoundError
    required = {"terratorch": "1.2.5", "torchgeo": "0.8.1"}
    installed = {}
    for package in required:
        try:
            installed[package] = version(package)
        except PackageNotFoundError:
            installed[package] = "not installed"
    if installed != required:
        raise RuntimeError(
            f"Incompatible notebook environment: {installed}. "
            "Run %pip install terratorch==1.2.5 torchgeo==0.8.1, "
            "restart the kernel, then Run All."
        )
    try:
        from terratorch.models.backbones.terramind.model.terramind_register import terramind_v1_large
    except ImportError as exc:
        raise ImportError(
            f"TerraTorch import failed with {installed}: {exc}. "
            "Inspect the underlying dependency error; restart the kernel after package changes."
        ) from exc
    # Keep the checkpoint's 224-grid positional embeddings; TerraMind interpolates them at forward time.
    backbone = terramind_v1_large(pretrained=False, modalities=["S1GRD"], img_size=224,
                                 merge_method="mean")
    state = torch.load(pretrained_path, map_location="cpu", weights_only=True, mmap=True)
    expected = backbone.state_dict()
    missing = [k for k in expected if k not in state]
    mismatched = [k for k in expected if k in state and expected[k].shape != state[k].shape]
    if missing or mismatched:
        raise RuntimeError(f"Incomplete backbone weights: missing={missing[:5]}, shape mismatch={mismatched[:5]}")
    # Ignore only unused modalities and the generative decoder; every selected encoder tensor must load.
    backbone.load_state_dict({k: state[k] for k in expected}, strict=True)
    backbone.requires_grad_(False).eval()
    print(f"Loaded all {len(expected)} S1 encoder tensors from the local checkpoint.")
    return backbone


class TerraMindFloodSegmentation(nn.Module):
    def __init__(self, terramind, head, encoder_batch_size=4):
        super().__init__()
        self.terramind = terramind.requires_grad_(False).eval()
        self.head = head
        self.encoder_batch_size = encoder_batch_size

    def train(self, mode=True):
        super().train(mode)
        self.terramind.eval()
        return self

    def forward(self, images):
        height, width = images.shape[-2:]
        ph, pw = (-height) % 16, (-width) % 16
        padded = F.pad(images, (0, pw, 0, ph), mode="replicate")
        gh, gw = padded.shape[-2] // 16, padded.shape[-1] // 16
        # Chunk only the frozen encoder to fit memory; keep the effective head batch unchanged.
        with torch.no_grad():
            features = [self.terramind({"S1GRD": chunk})[-1]
                        for chunk in padded.split(self.encoder_batch_size)]
            tokens = torch.cat(features, dim=0)
        if tokens.shape[1:] != (gh * gw, 1024):
            raise ValueError(f"Unexpected final TerraMind token shape: {tuple(tokens.shape)}")
        maps = tokens.transpose(1, 2).reshape(images.shape[0], 1024, gh, gw)
        logits = self.head(maps)
        logits = F.interpolate(logits, size=padded.shape[-2:], mode="bilinear", align_corners=False)
        return logits[..., :height, :width]


#### **First things first**


In [4]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

pretrained_path = "pretrained/terramind/TerraMind_v1_large.pt"

path_to_the_dataset = "datasets/sen1floods11/sen1floods11_hand_labeled_only/HandLabeled/"

path_to_the_dataset_index = "datasets/sen1floods11/sen1floods11_hand_labeled_only/flood_handlabeled/"

path_to_X = path_to_the_dataset + "S1Hand/"

path_to_Y = path_to_the_dataset + "LabelHand/"

path_to_train_csv = path_to_the_dataset_index + "flood_train_data.csv"

path_to_val_csv = path_to_the_dataset_index + "flood_valid_data.csv"

path_to_test_csv = path_to_the_dataset_index + "flood_test_data.csv"
path_to_test_bolivia_csv = path_to_the_dataset_index + "flood_bolivia_data.csv"

print(device)
split_manifest = audit_splits(path_to_the_dataset_index, path_to_X, path_to_Y)
print({k: v["count"] for k, v in split_manifest.items()})


cuda
{'train': 252, 'valid': 89, 'non_bol': 90, 'bol': 15}


In [5]:
CROP_SIZE = 120
IGNORE_INDEX = 255

#### **Functions**

In [6]:
def preprocess_train(data):
    """
    Normalize the complete image,
    extract a random 120x120 crop,
    and apply synchronized random flips.
    """

    image, mask = prepare_sample(data)

    H, W = image.shape[-2:]

    assert H >= CROP_SIZE and W >= CROP_SIZE

    # Random crop coordinates
    top = torch.randint(
        0, H - CROP_SIZE + 1, (1,), generator=crop_generator
    ).item()

    left = torch.randint(
        0, W - CROP_SIZE + 1, (1,), generator=crop_generator
    ).item()

    # Same crop for image and ground-truth mask
    image = image[
        :, top:top + CROP_SIZE, left:left + CROP_SIZE
    ]

    mask = mask[
        top:top + CROP_SIZE, left:left + CROP_SIZE
    ]

    # Random horizontal flip
    if torch.rand((), generator=crop_generator) < 0.5:
        image = image.flip(-1)
        mask = mask.flip(-1)

    # Random vertical flip
    if torch.rand((), generator=crop_generator) < 0.5:
        image = image.flip(-2)
        mask = mask.flip(-2)

    return image, mask



def preprocess_eval(data):
   
    return prepare_sample(data)

#### Creating datasets

In [7]:
train_data = load_split_paths(path_to_train_csv, path_to_X, path_to_Y)
val_data = load_split_paths(path_to_val_csv, path_to_X, path_to_Y)

In [8]:
print(f"Train samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")

Train samples: 252
Validation samples: 89


In [9]:
print(train_data[0])

(WindowsPath('datasets/sen1floods11/sen1floods11_hand_labeled_only/HandLabeled/S1Hand/Ghana_103272_S1Hand.tif'), WindowsPath('datasets/sen1floods11/sen1floods11_hand_labeled_only/HandLabeled/LabelHand/Ghana_103272_LabelHand.tif'))


In [10]:
train_dataset = Sen1Floods11Dataset(
    data_list=train_data,
    preprocess_func=preprocess_train
)

valid_dataset = Sen1Floods11Dataset(
    data_list=val_data,
    preprocess_func=preprocess_eval
)

#### Creating DataLoaders

In [11]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    generator=loader_generator,
    num_workers=0,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# Training batch checks

images, masks = next(iter(train_loader))

print("Training images:", images.shape)
print("Training masks:", masks.shape)

print("Image min:", images.min().item())
print("Image max:", images.max().item())

print("Water pixels:", (masks == 1).sum().item())
print("Non-water pixels:", (masks == 0).sum().item())
print("Ignored pixels:", (masks == IGNORE_INDEX).sum().item())

assert torch.isfinite(images).all()

assert set(torch.unique(masks).tolist()).issubset(
    {0, 1, IGNORE_INDEX}
)

# Validation batch checks

val_images, val_masks = next(iter(valid_loader))

print("-" * 40)
print("Validation images:", val_images.shape)
print("Validation masks:", val_masks.shape)

Training images: torch.Size([16, 2, 120, 120])
Training masks: torch.Size([16, 120, 120])
Image min: -6.461766719818115
Image max: 4.163671493530273
Water pixels: 31655
Non-water pixels: 167424
Ignored pixels: 31321
----------------------------------------
Validation images: torch.Size([1, 2, 512, 512])
Validation masks: torch.Size([1, 512, 512])


##### Loading TerraMind

In [12]:
terramind = load_terramind(pretrained_path).to(device)


ImportError: Install terratorch==1.2.5 in this notebook kernel, restart it, then Run All.

In [ ]:
terramind.requires_grad_(False) # Freeze the weights of TerraMind
terramind.eval()

### Creating the head

In [ ]:
head = FCNHead(in_channels=1024, channels=2)

head[1] = nn.GroupNorm(
    num_groups=16,
    num_channels=256
)

head = head.to(device)

In [ ]:
model = TerraMindFloodSegmentation(terramind, head).to(device)

In [ ]:
LR = 5e-4

class_weights = torch.tensor(
    [1.0, 8.0],
    dtype=torch.float32,
    device=device
)

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    ignore_index=IGNORE_INDEX
)

optimizer = torch.optim.AdamW(
    model.head.parameters(),
    lr=LR,
    weight_decay=1e-2
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=len(train_loader) * 10,
    T_mult=2,
    eta_min=0
)

In [ ]:
def train_one_epoch(model, train_loader, optimizer, criterion, device):

    model.train()

    total_loss = 0.0
    total_weight = 0

    for images, masks in train_loader:

        # Move batch to GPU
        images = images.to(device)
        masks = masks.to(device)

        valid = masks != IGNORE_INDEX

        if not valid.any():
            continue

        # Reset previous gradients
        optimizer.zero_grad(set_to_none=True)

        # Forward pass
        logits = model(images)

        # Compute segmentation loss
        loss = criterion(logits, masks)

        # Backpropagation
        loss.backward()

        # Update FCN head weights
        optimizer.step()
        
        # Scheduler step
        scheduler.step()
        
        # Correct denominator of the weighted mean
        batch_weight = criterion.weight[masks[valid]].sum().item()

        total_loss += loss.item() * batch_weight
        total_weight += batch_weight

    if total_weight == 0:
        raise RuntimeError("No valid training pixels.")

    return total_loss / total_weight

In [ ]:
@torch.inference_mode()
def sliding_window_logits(
    model,
    image,
    device,
    tile_size=120,
    stride=60,
    tile_batch_size=8
):
    """
    Predict segmentation logits for one complete image.

    Parameters
    ----------
    model:
        Frozen TerraMind + trained FCN head.

    image:
        Tensor [2, H, W], already normalized.

    device:
        CUDA or CPU.

    tile_size:
        Spatial size of each TerraMind input (120).

    stride:
        Distance between consecutive windows (60 = 50% overlap).

    tile_batch_size:
        Number of windows processed simultaneously.

    Returns
    -------
    logits:
        Tensor [2, H, W] on CPU.
    """

    model.eval()

    C, H, W = image.shape

    assert C == 2
    assert H >= tile_size and W >= tile_size
    assert 0 < stride <= tile_size

    # Generate starting positions along one dimension
    def get_starts(length):
        starts = list(
            range(0, length - tile_size + 1, stride)
        )

        # Ensure the final window reaches the image boundary
        if starts[-1] != length - tile_size:
            starts.append(length - tile_size)

        return starts

    # All window coordinates
    positions = [
        (top, left)
        for top in get_starts(H)
        for left in get_starts(W)
    ]

    # Accumulate predictions and coverage counts
    logits_sum = torch.zeros((2, H, W), dtype=torch.float32)
    counts = torch.zeros((H, W), dtype=torch.float32)

    # Process windows in batches
    for start in range(0, len(positions), tile_batch_size):

        batch_positions = positions[start:start + tile_batch_size]

        # Extract tiles
        tiles = torch.stack([
            image[:, top:top + tile_size, left:left + tile_size]
            for top, left in batch_positions
        ])

        tiles = tiles.to(device)

        # Run frozen TerraMind + FCN head
        outputs = model(tiles).cpu()

        # Place tile predictions back into the full image
        for (top, left), logits in zip(batch_positions, outputs):

            logits_sum[
                :, top:top + tile_size, left:left + tile_size
            ] += logits

            counts[
                top:top + tile_size, left:left + tile_size
            ] += 1

    assert (counts > 0).all(), "Some pixels were not covered."

    # Average logits wherever windows overlap
    return logits_sum / counts.unsqueeze(0)


import torch.nn.functional as F

@torch.inference_mode()
def validate(model, valid_loader, criterion, device):

    model.eval()

    total_loss = 0.0
    total_valid_pixels = 0

    TP = FP = FN = TN = 0

    for images, masks in valid_loader:

        # Process every complete image in the batch.
        for image, mask in zip(images, masks):
            mask = mask.cpu()
            
            
            valid = mask != IGNORE_INDEX

            if not valid.any():
                continue

            # Reconstruct logits using overlapping windows.
            logits = sliding_window_logits(
                model,
                image,
                device,
                tile_size=CROP_SIZE,
                stride=CROP_SIZE // 2,
                tile_batch_size=8
            )
            
            logits_device = logits.unsqueeze(0).to(device)
            mask_device = mask.unsqueeze(0).to(device=device, dtype=torch.long)

            loss = criterion(logits_device, mask_device)

            image_weight = criterion.weight[
                mask_device[mask_device != IGNORE_INDEX]
            ].sum().item()

            total_loss += loss.item() * image_weight
            total_valid_pixels += image_weight

            prediction = logits.argmax(dim=0)

            TP += ((prediction == 1) & (mask == 1) & valid).sum().item()
            FP += ((prediction == 1) & (mask == 0) & valid).sum().item()
            FN += ((prediction == 0) & (mask == 1) & valid).sum().item()
            TN += ((prediction == 0) & (mask == 0) & valid).sum().item()

    if total_valid_pixels == 0:
        raise RuntimeError("No valid validation pixels.")

    val_loss = total_loss / total_valid_pixels

    return {"val_loss": val_loss, **binary_metrics(TP, FP, FN, TN)}


In [ ]:
metrics = validate(model, valid_loader, criterion, device)

print(f"Validation Loss:      {metrics['val_loss']:.4f}")
print(f"Validation Water IoU: {metrics['water_iou']:.4f}")
print(f"Validation mIoU:      {metrics['miou']:.4f}")

In [ ]:
EPOCHS = 100
best_val_iou = -1.0
best_epoch = None
# Restart crop/shuffle RNGs after preview/baseline cells for repeatable execution.
crop_generator.manual_seed(SEED)
loader_generator.manual_seed(SEED)
pretrained_sha256 = sha256_file(pretrained_path)
run_config = {
    "schema": SCHEMA, "modality": "S1GRD", "seed": SEED, "epochs": EPOCHS,
    "eval_interval": 1, "preprocessing": PREPROCESSING, "splits": split_manifest,
    "pretrained_sha256": pretrained_sha256,
    "versions": {key: version(key) for key in ["torch", "torchvision", "terratorch", "torchgeo", "rasterio"]},
    "backbone_frozen": True, "feature_layer": 24, "feature_grid_for_120_crop": [8, 8],
    "head": "torchvision_FCNHead_1024_256_2_GroupNorm16",
    "batch_size": 16, "encoder_microbatch_size": 4, "precision": "fp32",
    "loss": "weighted_cross_entropy", "class_weights": [1.0, 8.0],
    "optimizer": "AdamW", "learning_rate": LR, "weight_decay": 0.01,
    "scheduler": {"name": "CosineAnnealingWarmRestarts", "T_0": len(train_loader)*10,
                  "T_mult": 2, "step_unit": "optimizer_update"},
    "checkpoint_selection": "validation_water_iou",
    "training_sampling": "uniform_random_120_crop_and_horizontal_vertical_flips",
    "all_ignore_policy": "retain_split_membership_skip_batches_with_no_valid_labels",
    "inference": {"tile_size": CROP_SIZE, "stride": 60, "aggregation": "mean_logits"},
    "comparison": "existing_CROMA_S1_FCN_recipe_with_model_native_normalization_and_patch_padding",
}

model_name = "terramind_large_s1_fcn.pt"

# A new directory is created each time this run cell is executed.
run_id = datetime.now().astimezone().strftime("%Y-%m-%d_%H-%M-%S_%f") + "_" + uuid4().hex[:8]
checkpoint_dir = Path("runs") / EXPERIMENT_NAME / f"train_{Path(model_name).stem}_{run_id}"
checkpoint_dir.mkdir(parents=True, exist_ok=False)
print(f"Run directory: {checkpoint_dir}")
checkpoint_path = checkpoint_dir / model_name
history_path = checkpoint_dir / "terramind_training_history.json"

with (checkpoint_dir / "run.json").open("w", encoding="utf-8") as run_file:
    json.dump({
        "experiment": EXPERIMENT_NAME,
        **run_config,
        "model_name": checkpoint_path.stem,
        "phase": "train",
        "run_id": run_id,
        "started_at": datetime.now().astimezone().isoformat(),
        "notebook": NOTEBOOK_PATH,
        "pretrained_weights": pretrained_path,
    "checkpoint": checkpoint_path.as_posix(),
    "history": history_path.as_posix(),
    }, run_file, indent=4)


history = {
    "train_loss": [],
    "val_loss": [],
    "val_water_iou": [],
    "val_miou": []
}

for epoch in range(EPOCHS):

    train_loss = train_one_epoch(
        model=model,
        criterion=criterion,
        train_loader=train_loader,
        device=device,
        optimizer=optimizer
    )

    metrics = validate(
        model=model,
        valid_loader=valid_loader,
        criterion=criterion,
        device=device
    )

    val_loss = metrics["val_loss"]
    val_water_iou = metrics["water_iou"]
    val_miou = metrics["miou"]

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_water_iou"].append(val_water_iou)
    history["val_miou"].append(val_miou)
    for key, value in metrics.items():
        if key not in {"val_loss", "water_iou", "miou"}:
            history.setdefault("val_" + key, []).append(value)
    history.setdefault("learning_rate", []).append(optimizer.param_groups[0]["lr"])

  
    if val_water_iou > best_val_iou:

        best_val_iou = val_water_iou
        best_epoch = epoch + 1

        torch.save(
            {
                "epoch": best_epoch,
                "run_config": run_config,
                "head_state_dict": model.head.state_dict(),
                "best_val_water_iou": best_val_iou,
                "val_miou": val_miou,
                "val_loss": val_loss,
            },
            checkpoint_path
        )
    
    with open(history_path, "w") as f:
        json.dump({
            "epochs_completed": epoch + 1,
            "best_epoch": best_epoch,
            "best_val_water_iou": best_val_iou,
            "history": history,
            "schema": SCHEMA,
            "run_config": run_config,
        }, f, indent=4)

 
    print(
        f"Epoch [{epoch+1:02d}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Water IoU: {val_water_iou:.4f} | "
        f"mIoU: {val_miou:.4f} | Precision: {metrics['precision']:.4f} | Recall: {metrics['recall']:.4f}"
    )

print(
    f"\nBest model: Epoch {best_epoch}, "
    f"Water IoU = {best_val_iou:.4f}"
)

In [ ]:
print(
    "TerraMind trainable parameters:",
    sum(p.numel() for p in model.terramind.parameters()
        if p.requires_grad)
)

print(
    "Head trainable parameters:",
    sum(p.numel() for p in model.head.parameters()
        if p.requires_grad)
)

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(10, 5))

plt.plot(
    epochs,
    history["train_loss"],
    label="Training loss",
    linewidth=2
)

plt.plot(
    epochs,
    history["val_loss"],
    label="Validation loss",
    linewidth=2
)

plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.title("TerraMind Large + FCN — Training and Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(checkpoint_dir / "training_loss.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    epochs,
    history["val_water_iou"],
    label="Water IoU",
    linewidth=2
)

plt.plot(
    epochs,
    history["val_miou"],
    label="Binary mIoU",
    linewidth=2
)

plt.axvline(
    best_epoch,
    linestyle="--",
    alpha=0.6,
    label=f"Best checkpoint (epoch {best_epoch})"
)

plt.xlabel("Epoch")
plt.ylabel("IoU")
plt.title("TerraMind Large + FCN — Validation Segmentation Performance")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(checkpoint_dir / "validation_iou.png", dpi=150, bbox_inches="tight")
plt.show()
from segmentation_metrics import plot_metric_history
plot_metric_history(history, checkpoint_dir)
plt.show()
